In [38]:
from transformers import BigBirdTokenizer
import pandas as pd
import os
from tqdm import tqdm
import json

In [39]:
model_id = 'google/bigbird-roberta-base'
tokenizer = BigBirdTokenizer.from_pretrained(model_id, trust_remote_code=True)

## Domain Adaptation Data

In [40]:
def convert_to_csv(input_path, output_path, tokenizer):
    assert os.path.exists(input_path)

    if not os.path.exists(output_path):
        os.makedirs(output_path)

    all_tokens_da = 0
    for train_or_dev in tqdm(os.listdir(input_path), 'Conversion to csv...'):
        train_or_dev_path = os.path.join(input_path, train_or_dev)

        df = {'entries': []}

        with open(train_or_dev_path, 'r') as f:
            objects = f.read().strip().split('\n')
            entries = [json.loads(obj) for obj in objects]

            raw_corpus = ""
            for entry in entries:
                if entry['exp_to_edit'] is None:
                    continue
                else:
                    if entry['exp_to_edit']:
                        if 'exp_upd' in entry and entry['exp_upd'] is not None:
                            raw_corpus += entry['exp_upd'] + '\n\n'
                            df['entries'].append(entry['exp_upd'])
                    else:
                        raw_corpus += entry['exp'] + '\n\n'
                        df['entries'].append(entry['exp'])

            raw_corpus = raw_corpus.strip()
            tokens_corpus = tokenizer.tokenize(raw_corpus)

            print(
                f'\n\n{train_or_dev} has {round(len(tokens_corpus) / (10 ** 6), 4)}M tokens for domain adaptation\n\n')

            all_tokens_da += len(tokens_corpus)

        df = pd.DataFrame(df)

        savename = 'dev.csv' if 'dev' in train_or_dev else 'train.csv'

        df.to_csv(os.path.join(output_path, savename), index=False)

    print(f'Total of {round(all_tokens_da / (10 ** 9), 4)}B tokens for domain adaptation')


convert_to_csv('raw_baseline_data', 'baseline_data', tokenizer)

Conversion to csv...:   0%|          | 0/2 [00:00<?, ?it/s]



train.jsonl has 18.2895M tokens for domain adaptation




Conversion to csv...: 100%|██████████| 2/2 [00:29<00:00, 14.74s/it]



dev.jsonl has 0.2603M tokens for domain adaptation


Total of 0.0185B tokens for domain adaptation


## QA Dataset

In [42]:
with open('raw_baseline_data/dev.jsonl', 'r') as f:
    lines = f.read().strip().split('\n')
    dev = [json.loads(line) for line in lines]

    test_data = []
    for test_example in tqdm(dev, desc='converting to eval QA dataset'):
        test_dict = {
            'question': test_example['question_upd'],
            'subject_name': test_example['subject_name'],
            'cop': test_example['cop'],
            'opa': test_example['opa_upd'],
            'opb': test_example['opb_upd'],
            'opc': test_example['opc_upd'],
            'opd': test_example['opd_upd']
        }
        test_data.append(test_dict)

with open('baseline_data/test.json', 'w') as fw:
    json.dump(test_data, fw, indent=2)

converting to eval QA dataset: 100%|██████████| 4183/4183 [00:00<00:00, 796837.75it/s]
